<a href="https://colab.research.google.com/github/mooch443/dataset-fixer/blob/main/notebooks/01_controlled_splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Controlled, group-aware dataset splitting

This tutorial demonstrates `Dataset.split()`: freezing physically related frames into groups, assigning those groups deterministically, previewing the proposal, and inspecting the resulting provenance.

> **AI-generation disclosure:** this project and tutorial are largely AI-generated under human direction and review. Independently validate results for your data.

The synthetic orchard images and annotations generated below are publicly available with this repository under the [MIT License](../LICENSE).

## 1. Install the package

The cell is idempotent in a fresh Colab runtime. Restart the runtime only if Colab asks after changing preinstalled dependencies.

In [ ]:
import os
if not os.path.isdir('/content/dataset-fixer'):
    !git clone -q https://github.com/mooch443/dataset-fixer.git /content/dataset-fixer
%cd /content/dataset-fixer
%pip install -q -e .

## 2. Generate and inspect the MIT-licensed example data

Frames with the same `row-XX` prefix represent one physical acquisition sequence. Keeping a sequence in one split prevents near-duplicate neighboring frames from leaking across train and validation.

In [ ]:
from pathlib import Path
from dataset_fixer import Dataset
from examples.create_example_datasets import create_example_datasets

example_root = Path('/content/dataset-fixer-examples')
paths = create_example_datasets(example_root, seed=42)
dataset = Dataset.open(paths['raw_sequences'], task='detect')
print(dataset)
print('classes:', dataset.classes)
dataset.visualize(split='train', n=9, seed=42, columns=3)

## 3. Split by physical sequence

`group_by` receives each source image path. The callback result becomes an indivisible allocation unit. A local seeded RNG makes the result reproducible, and `visualize=True` creates both the pre-operation sanity check and final distribution audit.

In [ ]:
import shutil
split_destination = Path('/content/orchard-grouped-split')
if split_destination.exists():
    shutil.rmtree(split_destination)

split_dataset = dataset.split(
    {'train': 0.70, 'val': 0.20, 'test': 0.10},
    group_by=lambda path: path.parent.name,
    seed=42,
    destination=split_destination,
    visualize=True,
)
print('name:', split_dataset.name)
print('location:', split_dataset.location)
print('data.yaml:', split_dataset.data_yaml)
print('splits:', split_dataset.splits)
print('training ready:', split_dataset.training_ready)

## 4. Verify the grouping and reproduction record

The assertion below is a useful pattern for automated dataset preparation: each physical group must map to exactly one output split. The manifest records the seed, resolved callback outputs, package revision, environment, and source fingerprint.

In [ ]:
import json
from collections import defaultdict

group_splits = defaultdict(set)
for record in split_dataset.provenance.values():
    group = Path(record['output_image']).parent.name
    group_splits[group].add(record['output_split'])
assert all(len(value) == 1 for value in group_splits.values())
print({group: next(iter(value)) for group, value in sorted(group_splits.items())})

manifest = json.loads((split_dataset.location / 'dataset-fixer.json').read_text())
print('settings fingerprint:', manifest['settings_fingerprint'])
print('tool revision:', manifest['environment']['dataset_fixer_git'])
print('provenance rows:', len(split_dataset.provenance))

### What to adapt for real data

- Change `group_by` to extract your orchard row, video, site, animal, patient, or sampling session from the path.
- Use `assign=` when specific groups have predetermined destinations. Conflicting assignments within one group fail before output is written.
- Use `deep=True` in `Dataset.open()` when byte-identical cross-split duplicates must also be detected.